<a href="https://colab.research.google.com/github/chetools/CHE4061_Spring2026/blob/main/PonchonSavarit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget -N -q https://raw.githubusercontent.com/chetools/chetools/main/tools/che5.ipynb -O che5.ipynb
%run che5.ipynb

In [2]:
p=Props(['Methanol','Isopropanol'])

In [3]:
def bubbleT_NRTL(x, P):

    T0 = np.sum(x*p.Tb(P))
    def froot(T):
        return np.sum(x*p.NRTL_gamma(x,T)*p.Pvap(T)/P)-1.

    T=sp.optimize.root_scalar(froot, x0=T0, method='secant').root
    return T, x*p.NRTL_gamma(x,T)*p.Pvap(T)/P

In [4]:
P=1e5
z1s = np.linspace(0,1,101)
y1s = []
bubbleTs = []
for x1 in z1s:
    T, (y1, _) = bubbleT_NRTL(np.array([x1, 1-x1]), P)
    bubbleTs.append(T)
    y1s.append(y1)
bubbleTs = np.array(bubbleTs)
y1s = np.array(y1s)

In [5]:
x_interp = sp.interpolate.PchipInterpolator(y1s, z1s)
Ty_interp = sp.interpolate.PchipInterpolator(y1s, bubbleTs)
Tx_interp = sp.interpolate.PchipInterpolator(z1s, bubbleTs)
Hvs = [p.Hv([y1, 1-y1], Ty_interp(y1)) for y1 in z1s]
Hv_interp =sp.interpolate.PchipInterpolator(z1s, Hvs)

Hls = [p.Hl([x1, 1-x1], Tx_interp(x1)) for x1 in z1s]
Hl_interp =sp.interpolate.PchipInterpolator(z1s, Hls)

In [19]:
F=1.
zF = 0.45
rD = 0.96
wD = rD*zF*F
xD = 0.98
D = wD/xD
B = F - D
wB = zF*F - wD
xB = wB/B

R = 5

Qc = (R*D + D) * (Hv_interp(xD)-Hl_interp(xD)) #condenser cooling
deltaD = Qc/D + Hl_interp(xD)

In [20]:
def froot_rec(y, x):
    Hv = Hv_interp(y)
    Hl = Hl_interp(x)
    return (deltaD - Hv)*(y-x) - (Hv - Hl)*(xD - y)



In [27]:
fig = make_subplots(rows=1,cols=1)
fig.add_scatter(x=z1s, y=Hv_interp(z1s), row=1,col=1)
fig.add_scatter(x=z1s, y=Hl_interp(z1s), row=1,col=1)
fig.add_scatter(x=[xD, xD], y=[Hl_interp(xD), deltaD], mode='lines', line_color='blue')

y=xD
for i in range(4):
    x = x_interp(y)
    fig.add_scatter(x=[y, x],y=[Hv_interp(y), Hl_interp(x)], mode='lines', line_color='rgba(100,100,100,0.5)', row=1, col=1)
    fig.add_scatter(x=[xD, x_interp(y)], y=[deltaD, Hl_interp(x)], mode='lines', line_color='blue', row=1, col=1)
    y=sp.optimize.root_scalar(lambda y: froot_rec(y, x), x0=x).root
    print(y)


fig.update_yaxes(range=[-3e5, 0.])
fig.update_layout(width=800, height=400)
fig

0.9586292939356538
0.9211044535970719
0.8616351532959992
0.780282472631463


In [ ]:
p.Hv([0.7,0.3], T)